In [50]:
import pandas as pd
from pathlib import Path

PLAYER_DIR = Path("../Datasets/Player_Statistics_2021-2026")

# The file names are inconsistent, so they are listed by hand: (general, defending, goalkeeper)
FILES = {
    "season-2122": ("player_stats_2021_2022.xlsx", "Player_Defending_stats_2021_2022.xlsx", "Goalkeeper_stats_2021_2022.xlsx"),
    "season-2223": ("player_stats_2022_2023.xlsx", "Player_Defending_stats_2022_2023.xlsx", "Goalkeeper_stats_2022_2023.xlsx"),
    "season-2324": ("player_stats_2023_2024.xlsx", "Player_Defending_stats_2023_2024.xlsx", "Goalkeeper_stats_2023_2024.xlsx"),
    "season-2425": ("player_stats24_25.xlsx",      "Player_Defending_stats_2024_2025.xlsx", "Goalkeeper_stats_2024_2025.xlsx"),
    "season-2526": ("Players_stat25_26.xls.xlsx",  "Player_Defending_stats_2025_2026.xlsx", "Goalkeeper_stats_2025_2026.xlsx"),
}

GENERAL_RENAME = {
    "Pos": "Position", "Squad": "Club", "Born": "Birth_Year",
    "MP": "Matches_Played", "Min": "Minutes", "90s": "Full_90s_Played",
    "Gls": "Goals", "Ast": "Assists", "G+A": "Goals_Plus_Assists", "G-PK": "Non_Penalty_Goals",
    "PK": "Penalties_Scored", "PKatt": "Penalties_Attempted",
    "CrdY": "Yellow_Cards", "CrdR": "Red_Cards",
    "Gls.1": "Goals_Per_90", "Ast.1": "Assists_Per_90", "G+A.1": "Goals_Plus_Assists_Per_90",
    "G-PK.1": "Non_Penalty_Goals_Per_90", "G+A-PK": "Non_Penalty_Goals_Plus_Assists_Per_90",
}
DEFENDING_RENAME = {
    "Squad": "Club", "Born": "Birth_Year", "2CrdY": "Second_Yellow_Cards",
    "Fls": "Fouls_Committed", "Fld": "Fouls_Drawn", "Off": "Offsides", "Crs": "Crosses",
    "Int": "Interceptions", "TklW": "Tackles_Won", "OG": "Own_Goals",
}
KEEPER_RENAME = {
    "Pos": "Position", "Squad": "Club", "Born": "Birth_Year",
    "MP": "Matches_Played", "Min": "Minutes", "90s": "Full_90s_Played",
    "GA": "Goals_Conceded", "GA90": "Goals_Conceded_Per_90", "SoTA": "Shots_On_Target_Faced",
    "Saves": "Saves", "Save%": "Save_Percentage", "W": "Wins", "D": "Draws", "L": "Losses",
    "CS": "Clean_Sheets", "CS%": "Clean_Sheet_Percentage",
    "PKatt": "Penalties_Faced", "PKA": "Penalties_Conceded", "PKsv": "Penalties_Saved",
    "PKm": "Penalties_Missed_By_Shooter", "Save%.1": "Penalty_Save_Percentage",
}

# Club names in the player files -> the names used in your team CSVs
CLUB_TO_TEAM = {
    "Manchester City": "Man City", "Manchester Utd": "Man United", "Nottingham": "Nott'm Forest",
    "Leeds United": "Leeds", "Leicester City": "Leicester", "Norwich City": "Norwich",
    "Luton Town": "Luton", "Ipswich Town": "Ipswich",
}

def fix_accents(text):
    """Repairs garbled names like 'ChÃ© Adams' -> 'Ché Adams' (the Defending files have this)."""
    if not isinstance(text, str):
        return text
    raw = bytearray()
    for ch in text:
        for enc in ("cp1252", "latin-1"):
            try:
                raw += ch.encode(enc)
                break
            except UnicodeEncodeError:
                continue
        else:
            return text
    try:
        return raw.decode("utf-8")
    except UnicodeDecodeError:
        return text

def read_table(path):
    df = pd.read_excel(path, header=1)                        # real headers are on the 2nd row
    df = df.drop(columns=["Rk", "Matches"], errors="ignore")  # row number and junk link column
    df["Player"] = df["Player"].map(fix_accents)
    return df

def load_season(season, files):
    general_file, defending_file, keeper_file = files
    general = read_table(PLAYER_DIR / general_file).rename(columns=GENERAL_RENAME)
    defending = read_table(PLAYER_DIR / defending_file).rename(columns=DEFENDING_RENAME)
    keepers = read_table(PLAYER_DIR / keeper_file).rename(columns=KEEPER_RENAME)

    extra_cols = ["Second_Yellow_Cards", "Fouls_Committed", "Fouls_Drawn", "Offsides",
                  "Crosses", "Interceptions", "Tackles_Won", "Own_Goals"]
    players = general.merge(
        defending[["Player", "Club", "Birth_Year"] + extra_cols],
        on=["Player", "Club", "Birth_Year"], how="left", validate="one_to_one",
    )

    for table in (players, keepers):
        table["Season"] = season
        table["Team"] = table["Club"].replace(CLUB_TO_TEAM)
        table["Main_Position"] = table["Position"].str.split(",").str[0]   # "MF,FW" -> "MF"
        table["Nation_Code"] = table["Nation"].str.split().str[-1]         # "eng ENG" -> "ENG"
    return players, keepers

parts = [load_season(season, files) for season, files in FILES.items()]
players = pd.concat([p for p, _ in parts], ignore_index=True)
keepers = pd.concat([k for _, k in parts], ignore_index=True)

print("players:", players.shape, "| keepers:", keepers.shape)   # expect (2820, 35) and (205, 29)

players: (2820, 35) | keepers: (205, 29)


In [46]:
# Player goals + own goals should equal the goals in the team CSVs
team_dir = Path("../Datasets/Premier_League_Team_Statistics_2021-2026")

for season in FILES:
    t = pd.read_csv(team_dir / f"{season}.csv")
    team_goals = t["FTHG"].sum() + t["FTAG"].sum()
    s = players[players["Season"] == season]
print(season, "team goals:", team_goals, "| player goals + own goals:", s["Goals"].sum() + s["Own_Goals"].sum())

season-2526 team goals: 1045 | player goals + own goals: 1045


In [47]:
goal_points = {"FW": 4, "MF": 5, "DF": 6, "GK": 6}   # points per goal depend on position

players["Fantasy_Points"] = (
    players["Full_90s_Played"] * 2                    # appearance points (approximate)
    + players["Goals"] * players["Main_Position"].map(goal_points)
    + players["Assists"] * 3
    - players["Yellow_Cards"] * 1
    - players["Red_Cards"] * 3
    - players["Own_Goals"] * 2
    - (players["Penalties_Attempted"] - players["Penalties_Scored"]) * 2   # missed penalties
)
players["Fantasy_Points_Per_90"] = players["Fantasy_Points"] / players["Full_90s_Played"]

# Top 5 per season
for season, g in players.groupby("Season"):
    print(season)
    print(g.nlargest(5, "Fantasy_Points")[["Player", "Club", "Main_Position", "Goals", "Assists", "Fantasy_Points"]].to_string(index=False), "\n")

season-2122
         Player            Club Main_Position  Goals  Assists  Fantasy_Points
  Son Heung-min       Tottenham            MF     23        7           200.8
  Mohamed Salah       Liverpool            FW     23       13           189.4
     Harry Kane       Tottenham            FW     17        9           161.8
   Jarrod Bowen        West Ham            MF     12       10           150.2
Kevin De Bruyne Manchester City            MF     15        8           146.0 

season-2223
         Player            Club Main_Position  Goals  Assists  Fantasy_Points
 Erling Haaland Manchester City            FW     36        8           224.6
     Harry Kane       Tottenham            FW     30        3           196.6
  Mohamed Salah       Liverpool            FW     19       12           179.2
Martin Ødegaard         Arsenal            MF     15        7           161.4
    Bukayo Saka         Arsenal            FW     14       11           151.6 

season-2324
        Player          

In [48]:
# Start from the keeper's general points (minutes, goals, assists, cards, own goals)
gk_base = players.loc[players["Main_Position"] == "GK", ["Season", "Player", "Club", "Fantasy_Points"]]

# Drop earlier versions so this cell can be re-run safely
keepers = keepers.drop(columns=["Base_Points", "Keeper_Points", "Keeper_Points_Per_90"], errors="ignore")
keepers = keepers.merge(
    gk_base.rename(columns={"Fantasy_Points": "Base_Points"}),
    on=["Season", "Player", "Club"], how="left", validate="one_to_one",
)

# Add the goalkeeper-specific points
keepers["Keeper_Points"] = (
    keepers["Base_Points"]
    + keepers["Clean_Sheets"] * 4          # clean sheet
    + keepers["Saves"] / 3                 # 1 point for every 3 saves
    + keepers["Penalties_Saved"] * 5       # saved penalty
    - keepers["Goals_Conceded"] / 2        # -1 for every 2 goals conceded
)
keepers["Keeper_Points_Per_90"] = keepers["Keeper_Points"] / keepers["Full_90s_Played"]

print("keepers:", keepers.shape)           # expect (205, 30)

for season, g in keepers.groupby("Season"):
    print(season)
    print(g.nlargest(3, "Keeper_Points")[["Player", "Club", "Clean_Sheets", "Saves", "Goals_Conceded", "Keeper_Points"]].round(1).to_string(index=False), "\n")

keepers: (205, 32)
season-2122
        Player            Club  Clean_Sheets  Saves  Goals_Conceded  Keeper_Points
       Alisson       Liverpool            20     75              24          166.0
Ederson Moraes Manchester City            20     58              26          157.3
   Hugo Lloris       Tottenham            16     97              40          150.3 

season-2223
      Player           Club  Clean_Sheets  Saves  Goals_Conceded  Keeper_Points
David de Gea Manchester Utd            17    100              43          160.8
     Alisson      Liverpool            14    104              43          150.2
  David Raya      Brentford            12    154              46          149.3 

season-2324
     Player           Club  Clean_Sheets  Saves  Goals_Conceded  Keeper_Points
André Onana Manchester Utd             9    148              58          127.3
 David Raya        Arsenal            14     45              24          126.0
 Bernd Leno         Fulham            10    134     

In [49]:
import numpy as np

# 1. Each team's clean sheets and goals conceded, from the team CSVs
team_dir = Path("../Datasets/Premier_League_Team_Statistics_2021-2026")

team_rows = []
for season in FILES:
    t = pd.read_csv(team_dir / f"{season}.csv")
    home = pd.DataFrame({"Season": season, "Team": t["HomeTeam"],
                         "Team_Goals_Conceded": t["FTAG"], "Team_Clean_Sheets": (t["FTAG"] == 0).astype(int)})
    away = pd.DataFrame({"Season": season, "Team": t["AwayTeam"],
                         "Team_Goals_Conceded": t["FTHG"], "Team_Clean_Sheets": (t["FTHG"] == 0).astype(int)})
    team_rows += [home, away]
team_defence = pd.concat(team_rows).groupby(["Season", "Team"], as_index=False).sum()

# 2. Add them to each player (dropping old versions first so this cell can be re-run)
players = players.drop(columns=["Team_Goals_Conceded", "Team_Clean_Sheets", "Minutes_Share", "Estimated_Clean_Sheets",
                                "Estimated_Goals_Conceded", "Defensive_Points", "Keeper_Points", "Total_Points"],
                       errors="ignore")
players = players.merge(team_defence, on=["Season", "Team"], how="left", validate="many_to_one")

# 3. Estimate what each player was on the pitch for: team total x share of the season's minutes
players["Minutes_Share"] = (players["Minutes"] / (38 * 90)).clip(upper=1)
players["Estimated_Clean_Sheets"] = players["Team_Clean_Sheets"] * players["Minutes_Share"]
players["Estimated_Goals_Conceded"] = players["Team_Goals_Conceded"] * players["Minutes_Share"]

# 4. Defender and midfielder points
defender_points = (players["Estimated_Clean_Sheets"] * 4
                   - players["Estimated_Goals_Conceded"] / 2
                   + (players["Tackles_Won"] + players["Interceptions"]) * 0.2)
midfielder_points = players["Estimated_Clean_Sheets"] * 1

players["Defensive_Points"] = np.select(
    [players["Main_Position"] == "DF", players["Main_Position"] == "MF"],
    [defender_points, midfielder_points], default=0)

# 5. One combined score: keepers use Keeper_Points, everyone else uses Fantasy_Points + Defensive_Points
players = players.merge(keepers[["Season", "Player", "Club", "Keeper_Points"]],
                        on=["Season", "Player", "Club"], how="left", validate="one_to_one")
players["Total_Points"] = np.where(players["Main_Position"] == "GK",
                                   players["Keeper_Points"],
                                   players["Fantasy_Points"] + players["Defensive_Points"])

print("players:", players.shape)   # expect (2820, 45)

# Top 3 defenders per season
for season, g in players[players["Main_Position"] == "DF"].groupby("Season"):
    print(season)
    print(g.nlargest(3, "Total_Points")[["Player", "Club", "Estimated_Clean_Sheets", "Tackles_Won", "Interceptions", "Total_Points"]].round(1).to_string(index=False), "\n")

players: (2820, 45)
season-2122
                Player            Club  Estimated_Clean_Sheets  Tackles_Won  Interceptions  Total_Points
Trent Alexander-Arnold       Liverpool                    17.5           24             44         182.2
          João Cancelo Manchester City                    19.8           53             56         180.6
            Joël Matip       Liverpool                    17.1           35             45         160.9 

season-2223
                Player      Club  Estimated_Clean_Sheets  Tackles_Won  Interceptions  Total_Points
       Kieran Trippier Newcastle                    13.7           48             43         151.0
Trent Alexander-Arnold Liverpool                    12.0           38             40         142.4
     Gabriel Magalhães   Arsenal                    14.0           28             32         135.2 

season-2324
           Player    Club  Estimated_Clean_Sheets  Tackles_Won  Interceptions  Total_Points
        Ben White Arsenal       